## Foundations of the Predictive Architecture
The success of a machine learning model depends on the quality of its environment. For our study in Brazzaville, we select specialized libraries that transform raw student profiles into mathematical insights. This digital ecosystem ensures the stability of our Gradient Boosting Machine while providing the metrics required for a rigorous validation. We rely on the following pillars to build our predictive oracle.
- **The gbm** library handles the iterative construction of decision trees by minimizing errors through successive gradients.
- **The caret package** provides a unified framework for data partitioning and hyperparameter tuning to ensure the model generalizes well to the Congolese context.
- **The pROC tool** enables the calculation of the Area Under the Curve to measure the classifier's ability to distinguish between academic success and failure.

In [6]:
# Installing the core Gradient Boosting Machine engine
install.packages("gbm")

# Installing the Classification and REgression Training framework
install.packages("caret")

# Installing the tools for Receiver Operating Characteristic analysis
install.packages("pROC")

# Installing the necessary library for Excel files
install.packages("readxl")

The following package(s) will be installed:
- gbm [2.2.3]
These packages will be installed into "C:/Users/bonhe/Repositories/gbm-learning-strategy/renv/library/windows/R-4.5/x86_64-w64-mingw32".

# Installing packages --------------------------------------------------------
✔ gbm 2.2.3                                [linked from cache]
The following package(s) will be installed:
- caret [7.0-1]
These packages will be installed into "C:/Users/bonhe/Repositories/gbm-learning-strategy/renv/library/windows/R-4.5/x86_64-w64-mingw32".

# Installing packages --------------------------------------------------------
✔ caret 7.0-1                              [linked from cache]
The following package(s) will be installed:
- pROC [1.19.0.1]
These packages will be installed into "C:/Users/bonhe/Repositories/gbm-learning-strategy/renv/library/windows/R-4.5/x86_64-w64-mingw32".

# Installing packages --------------------------------------------------------
✔ pROC 1.19.0.1                            

In [5]:
# Loading the libraries into the current Jupyter session
library(gbm)
library(caret)
library(pROC)
library(readxl)

### Preparing the dataset

In [9]:
# Importing the Excel file into a data frame named 'student_data'
# We ensure the file path matches your local directory structure

student_data <- read_excel("data_src.xlsx")


# Identifying the number of missing values (NA) per column
# This overview highlights variables with data collection gaps

colSums(is.na(student_data))

# Strategy for numeric variables: Replacing NAs with the median
# This maintains the distribution without losing valuable student profiles

for(i in 1:ncol(student_data)) {
  if(is.numeric(student_data[[i]])) {
    student_data[[i]][is.na(student_data[[i]])] <- median(student_data[[i]], na.rm = TRUE)
  }
}

# Final check to ensure the dataset is now complete
# sum(is.na(student_data))

# Assuming 'Pass' becomes 1 and 'Fail' becomes 0
# You should adapt the strings "Pass" and "Fail" to match your exact data entries
student_data$Result_Binary <- ifelse(student_data$Result == "Pass", 1, 0)

# Processing Binary Variables
# Converting the gender variable into a binary indicator for mathematical stability

student_data$gender <- ifelse(student_data$gender == "Male", 1, 0)

# Managing Missing Values for the Device Column
# We fill gaps with "None" to represent students without digital access

student_data$device[is.na(student_data$device)] <- "None"

# Implementing One-Hot Encoding for Multi-Category Factors
# This method expands the matrix to include specific vectors for every category

# Selecting variables for the transformation based on the actual dataset headers
categorical_vars <- c("language", "stream", "device", "Independent_study", "class_size")

# Using the dummyVars function to create the expansion model
encoder_model <- dummyVars(~ language + stream + device + Independent_study + class_size, data = student_data)
encoded_features <- data.frame(predict(encoder_model, newdata = student_data))

# Merging the new numerical vectors with the main dataset
# We drop the original text columns to ensure the final matrix is purely numerical

columns_to_remove <- c(categorical_vars, "student_id", "school_id", "Result", "sn")
student_data_final <- cbind(student_data[, !(names(student_data) %in% columns_to_remove)], encoded_features)

# Final structural audit to confirm the numerical readiness
str(student_data_final)




sn                  student_id 
                          0                           0 
                  school_id     power_cut_hour_per_week 
                          0                           0 
            study_during_pc      study_quiet_place_1_10 
                          0                           0 
study_interruption_per_hour        sleep_hour_per_night 
                          0                           0 
         sleep_quality_1_10                 stress_1_10 
                          0                           0 
          technostress_1_10         cognitive_load_1_10 
                          0                           0 
            resilience_1_10 digital_education_hour_week 
                          0                           0 
      social_media_hour_day                      device 
                          0                           0 
             ai_use_req_1_5            commute_hour_day 
                          0                           0 
    hour_searching_per_week           active_recall_1_5 
                          0                           0 
         self_test_per_week           Independent_study 
                          0                           0 
    past_test_hour_per_week                       tutor 
                          0                           0 
        tutor_hour_per_week           spaced_repetition 
                          0                           0 
               absence_days    instruction_clarity_1_10 
                          0                           0 
                     gender                         age 
                          0                           0 
                   language                      stream 
                          0                           0 
                 class_size          chore_hour_per_day 
                          0                           0 
                 mock_score                      Result 
                          0                           0

'data.frame':	250 obs. of  47 variables:
 $ power_cut_hour_per_week    : num  58 70 16 38 48 1 64 26 45 80 ...
 $ study_during_pc            : num  0 0 1 0 0 1 0 1 0 0 ...
 $ study_quiet_place_1_10     : num  4 2 10 4 4 10 2 8 4 2 ...
 $ study_interruption_per_hour: num  4 5 0 3 4 0 6 1 4 8 ...
 $ sleep_hour_per_night       : num  5.5 4.8 7.8 6.2 5.8 8.2 4.5 7 5.4 4 ...
 $ sleep_quality_1_10         : num  4 2 10 6 4 10 2 8 4 2 ...
 $ stress_1_10                : num  8 9 3 7 7 2 9 4 8 10 ...
 $ technostress_1_10          : num  4 5 1 3 4 1 5 2 4 5 ...
 $ cognitive_load_1_10        : num  7 8 3 6 7 2 9 4 8 9 ...
 $ resilience_1_10            : num  5 4 9 7 6 10 3 9 5 2 ...
 $ digital_education_hour_week: num  4 2 12 5 4 15 2 10 3 1 ...
 $ social_media_hour_day      : num  3 4.5 1.2 2.5 3.2 1 5 1.5 3.5 5.5 ...
 $ ai_use_req_1_5             : num  2 1 3 2 1 4 1 3 1 1 ...
 $ commute_hour_day           : num  2 2.5 0.5 1.5 1.8 0.3 3 0.8 2.2 3.2 ...
 $ hour_searching_per_week    : num  60 9

### Standardizing Metrics  
We use Z‑score scaling to place variables like power cuts and commute times on comparable scales. This ensures no single measure dominates the model simply due to its magnitude, allowing each factor to contribute fairly to predictions while preserving the original data distribution.

In [10]:
# Establishing the exhaustive list of quantitative variables for the Brazzaville study
# This list includes all metrics that are neither binary nor categorical flags

quantitative_vars <- c(
  "power_cut_hour_per_week", "study_quiet_place_1_10", "study_interruption_per_hour",
  "sleep_hour_per_night", "sleep_quality_1_10", "stress_1_10", "technostress_1_10",
  "cognitive_load_1_10", "resilience_1_10", "digital_education_hour_week",
  "social_media_hour_day", "ai_use_req_1_5", "commute_hour_day", 
  "hour_searching_per_week", "active_recall_1_5", "self_test_per_week",
  "past_test_hour_per_week", "tutor_hour_per_week", "absence_days", 
  "instruction_clarity_1_10", "age", "chore_hour_per_day", "mock_score"
)

# Executing the standardization to remove unit-based biases
# The scale function ensures that each variable contributes fairly to the predictive logic

student_data_final[quantitative_vars] <- scale(student_data_final[quantitative_vars])

# Confirming the statistical normalization of the dataset
# We verify that the mean of each quantitative variable is now zero

summary(student_data_final[quantitative_vars])

 power_cut_hour_per_week study_quiet_place_1_10 study_interruption_per_hour
 Min.   :-1.7429         Min.   :-1.2030        Min.   :-1.31750           
 1st Qu.:-0.7000         1st Qu.:-0.4783        1st Qu.:-0.90629           
 Median :-0.1994         Median :-0.4783        Median :-0.08389           
 Mean   : 0.0000         Mean   : 0.0000        Mean   : 0.00000           
 3rd Qu.: 0.3116         3rd Qu.: 0.9711        3rd Qu.: 0.32732           
 Max.   : 3.5551         Max.   : 1.6958        Max.   : 2.38334           
 sleep_hour_per_night sleep_quality_1_10  stress_1_10      technostress_1_10
 Min.   :-2.8185      Min.   :-1.2718    Min.   :-1.8117   Min.   :-1.6220  
 1st Qu.:-0.5329      1st Qu.:-0.5409    1st Qu.:-0.9837   1st Qu.:-0.8820  
 Median : 0.1306      Median : 0.1900    Median : 0.2583   Median :-0.1421  
 Mean   : 0.0000      Mean   : 0.0000    Mean   : 0.0000   Mean   : 0.0000  
 3rd Qu.: 0.7942      3rd Qu.: 0.9209    3rd Qu.: 0.6724   3rd Qu.: 0.5979  
 Max. 